### Базовая установка (Colab) + клонирование репозитория

In [1]:
!pip -q install -U wandb psutil faiss-cpu

# PyG — ставим подходящие колеса
import torch, subprocess, sys
torch_ver = torch.__version__.split('+')[0]
cuda = torch.version.cuda
if cuda is None:
    url = f"https://data.pyg.org/whl/torch-{torch_ver}+cpu.html"
else:
    url = f"https://data.pyg.org/whl/torch-{torch_ver}+cu{cuda.replace('.','')}.html"
print("torch:", torch.__version__, "cuda:", cuda, "pyg-url:", url)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-index",
                       "torch-scatter", "torch-sparse", "torch-cluster", "torch-spline-conv", "-f", url])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch-geometric"])

# Clone repo
!rm -rf /kaggle/working/vseros_b
!git clone -q https://github.com/Icedarold/vseros_b /kaggle/working/vseros_b

import sys, os
sys.path.append("/kaggle/working/vseros_b/src")
print("Repo ready:", os.listdir("/kaggle/working/vseros_b"))

torch: 2.6.0+cu124 cuda: 12.4 pyg-url: https://data.pyg.org/whl/torch-2.6.0+cu124.html
Repo ready: ['.git', 'src', '.gitignore', 'README.md', 'LICENSE', 'notebooks']


In [12]:
from pathlib import Path
from vseros_b.config import PATHS

# ЗАМЕНИ на свой датасет-слег
DATASET_ROOT = Path("/kaggle/input/vseros-recsys-osn")  # <-- поменяй
PATHS.train_path  = DATASET_ROOT / "train_data.pq"
PATHS.sample_path = DATASET_ROOT / "sample_submission.csv"

# Директории для артефактов/сабмитов в working (они сохранятся в Output при Save Version)
PATHS.root_dir    = Path("/kaggle/working/vseros_b")
PATHS.artifact_dir= PATHS.root_dir / "artifacts"
PATHS.cand_dir    = PATHS.root_dir / "artifacts" / "candidates"
PATHS.metrics_dir = PATHS.root_dir / "artifacts" / "metrics"
PATHS.sub_dir     = PATHS.root_dir / "submissions"

from vseros_b.artifacts import ensure_project_dirs
ensure_project_dirs()
print("Paths set:\n train:", PATHS.train_path, "\n sample:", PATHS.sample_path, "\n out:", PATHS.root_dir)

Paths set:
 train: /kaggle/input/vseros-recsys-osn/train_data.pq 
 sample: /kaggle/input/vseros-recsys-osn/sample_submission.csv 
 out: /kaggle/working/vseros_b


### Импорты проекта + W&B init (глобальный ран-«оркестратор»)

In [13]:
import os
import wandb
from kaggle_secrets import UserSecretsClient

# Получаем доступ к секретам
user_secrets = UserSecretsClient()

# Получаем ваш WANDB_API_KEY из Kaggle Secrets
# Убедитесь, что вы назвали секрет именно "WANDB_API_KEY"
try:
    api_key = user_secrets.get_secret("WANDB_API_KEY")
    wandb.login(key=api_key)
    print("W&B logged in successfully.")
except Exception as e:
    print("Could not log in to W&B. Check your API key and internet connection.")
    print(f"Error: {e}")
    # Если логин не удался, можно попробовать анонимный режим или остановить выполнение
    # wandb.login(anonymous="allow")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


W&B logged in successfully.


### Загрузка и подготовка данных (train/val + кэши)

In [14]:
from vseros_b.data import load_and_prepare
ctx = load_and_prepare(data_path=PATHS.train_path)

print({
    "rows/train": len(ctx["train_df"]),
    "rows/val": len(ctx["val_df"]),
    "date_min": ctx["split"].date_min,
    "train_end": ctx["split"].train_end,
    "val_start": ctx["split"].val_start,
    "val_end": ctx["split"].val_end,
})

{'rows_total': 8777975, 'rows_train': 7326431, 'rows_val': 1451544, 'users': 2682603, 'items': 740651, 'date_min': 0, 'date_max': 46, 'val_range': [40, 46]}
{'rows/train': 7326431, 'rows/val': 1451544, 'date_min': 0, 'train_end': 39, 'val_start': 40, 'val_end': 46}


### Хелпер для запуска эксперимента (единый интерфейс)

In [15]:
# Cell 3 — experiment runner helper
def run_experiment(exp_cls, exp_name: str, **init_kwargs):
    """
    exp_cls — класс эксперимента (наследник BaseExperiment),
    exp_name — строковое имя для W&B job_type и логов,
    init_kwargs — параметры конструктора эксперимента.
    """
    try:
        exp = exp_cls(**init_kwargs)
    except Exception as e:
        print(f"[{exp_name}] init failed:", e)
        return None

    exp_run = wandb.init(
        project=WANDB_PROJECT,
        group=WANDB_GROUP,
        job_type=exp_name,
        name=exp_name,
        reinit=True,
    )
    try:
        print(f"[{exp_name}] fit()…")
        state = exp.fit(ctx)

        print(f"[{exp_name}] evaluate()…")
        metrics = exp.evaluate(ctx)

        print(f"[{exp_name}] save()…")
        saved = exp.save(ctx)

        # короткий лог в summary
        if isinstance(metrics, pd.DataFrame) and not metrics.empty:
            # пример: взять первое/лучшее значение
            try:
                wandb.summary[f"{exp_name}/metrics_rows"] = int(len(metrics))
            except Exception:
                pass
        print(f"[{exp_name}] done.")
        return exp, state, metrics, saved
    except Exception as e:
        print(f"[{exp_name}] FAILED with error:\n", e)
        return None
    finally:
        wandb.finish()


### REGISTRY: какие эксперименты запускать

In [16]:
# Cell 4 — registry (импорт и регистрация всех экспериментов)
import os, glob
from pathlib import Path
from typing import Optional

from vseros_b.config import PATHS

REGISTRY = []

def _pick_largest(pattern: Path) -> Optional[Path]:
    """Вернёт самый большой по размеру файл по шаблону или None."""
    files = glob.glob(str(pattern))
    if not files:
        return None
    files = sorted(files, key=lambda p: os.path.getsize(p))
    return Path(files[-1])

def _auto_lightgcn_import_config():
    """Автопоиск артефактов LightGCN: user_emb_dim*_L*.npy, item_emb_dim*_L*.npy и mapping-и."""
    from vseros_b.lightgcn_import import ImportConfig
    root = PATHS.artifact_dir / "lightgcn"
    user_emb = _pick_largest(root / "user_emb_dim*_L*.npy")
    item_emb = _pick_largest(root / "item_emb_dim*_L*.npy")
    user_map = root / "user_map.parquet"
    item_map = root / "item_map.parquet"
    if user_emb and item_emb and user_map.exists() and item_map.exists():
        return ImportConfig(
            user_emb_path=user_emb,
            item_emb_path=item_emb,
            user_map_path=user_map,
            item_map_path=item_map,
            use_faiss=True,
        )
    # дефолт — попробуем стандартные имена, если вдруг они есть
    return ImportConfig(use_faiss=True)

def _auto_candidates_dir(exp_name: str) -> Optional[Path]:
    """Вернёт последний по размеру файл с кандидатами внутри директории эксперимента."""
    root = PATHS.cand_dir / exp_name
    return _pick_largest(root / "val_candidates*.parquet")

# ---------------- exp101: pop_decay ----------------
# try:
#     from vseros_b.exp101_pop_decay import Exp101PopDecay, Exp101Config
#     REGISTRY.append(("exp101_pop_decay", Exp101PopDecay, {"cfg": Exp101Config()}))
#     print("[registry] exp101 registered")
# except Exception as e:
#     print("[registry] exp101 missing:", e)

# ---------------- exp102: daily_trending ----------------
# try:
#     from vseros_b.exp102_daily_trending import Exp102DailyTrending, Exp102Config
#     REGISTRY.append(("exp102_daily_trending", Exp102DailyTrending, {"cfg": Exp102Config(mode="frozen_train")}))
#     print("[registry] exp102 registered")
# except Exception as e:
#     print("[registry] exp102 missing:", e)

# # ---------------- exp103: covis v1 ----------------
# try:
#     from vseros_b.exp103_covis_v1 import Exp103CoVisV1, Exp103Config
#     REGISTRY.append(("exp103_covis_v1", Exp103CoVisV1, {"cfg": Exp103Config()}))
#     print("[registry] exp103 registered")
# except Exception as e:
#     print("[registry] exp103 missing:", e)

# # ---------------- exp104: covis 2hop ----------------
# try:
#     from vseros_b.exp104_covis_2hop import Exp104CoVis2Hop, Exp104Config
#     REGISTRY.append(("exp104_covis_2hop", Exp104CoVis2Hop, {"cfg": Exp104Config()}))
#     print("[registry] exp104 registered")
# except Exception as e:
#     print("[registry] exp104 missing:", e)

# # ---------------- exp105: item2vec ----------------
# try:
#     from vseros_b.exp105_item2vec import Exp105Item2Vec, Exp105Config
#     REGISTRY.append(("exp105_item2vec", Exp105Item2Vec, {"cfg": Exp105Config()}))
#     print("[registry] exp105 registered")
# except Exception as e:
#     print("[registry] exp105 missing:", e)

# ---------------- exp106: lightgcn_import ----------------
try:
    from vseros_b.exp106_lightgcn_import import Exp106LightGCNImport, Exp106Config
    from vseros_b.lightgcn_import import ImportConfig
    imp_cfg = _auto_lightgcn_import_config()
    cfg106 = Exp106Config(import_cfg=imp_cfg, pool_mode="popular", pool_K=3000, m_per_user=1000)
    REGISTRY.append(("exp106_lightgcn_import", Exp106LightGCNImport, {"cfg": cfg106}))
    print("[registry] exp106 registered")
except Exception as e:
    print("[registry] exp106 missing:", e)

# ---------------- exp107: ppr ----------------
try:
    from vseros_b.exp107_ppr import Exp107PPR, Exp107Config
    REGISTRY.append(("exp107_ppr", Exp107PPR, {"cfg": Exp107Config()}))
    print("[registry] exp107 registered")
except Exception as e:
    print("[registry] exp107 missing:", e)

# ---------------- exp108: recall_fusion ----------------
try:
    from vseros_b.exp108_recall_fusion import Exp108RecallFusion, Exp108Config, SourceSpec

    # Автосбор источников из уже сохранённых кандидатов
    src_specs = []
    for name, exp_dir in [
        ("pop_decay", "exp101_pop_decay"),
        ("trending",  "exp102_daily_trending"),
        ("covis_v1",  "exp103_covis_v1"),
        ("covis_2h",  "exp104_covis_2hop"),
        ("item2vec",  "exp105_item2vec"),
        ("lgcn",      "exp106_lightgcn_import"),
        ("ppr",       "exp107_ppr"),
    ]:
        p = _auto_candidates_dir(exp_dir)
        if p:
            src_specs.append(SourceSpec(name=name, path=p))
        else:
            print(f"[registry] fusion source not found: {name} ({exp_dir})")

    weights = {
        "pop_decay": 0.8,
        "trending":  1.0,
        "covis_v1":  1.2,
        "covis_2h":  1.0,
        "item2vec":  1.0,
        "lgcn":      1.5,
        "ppr":       1.2,
    }

    cfg108 = Exp108Config(
        sources=src_specs,
        weights=weights,
        score_scheme="rr",
        cap_per_source=400,
        m_per_user=1000,
        backfill_with_global_top=True,
        backfill_K=3000,
    )
    REGISTRY.append(("exp108_recall_fusion", Exp108RecallFusion, {"cfg": cfg108}))
    print("[registry] exp108 registered")
except Exception as e:
    print("[registry] exp108 missing:", e)

print(f"[registry] total registered: {len(REGISTRY)}")

[registry] exp102 registered
[registry] exp103 registered
[registry] exp104 registered
[registry] exp105 registered
[registry] exp106 missing: mutable default <class 'vseros_b.lightgcn_import.ImportConfig'> for field import_cfg is not allowed: use default_factory
[registry] exp107 registered
[registry] fusion source not found: pop_decay (exp101_pop_decay)
[registry] fusion source not found: trending (exp102_daily_trending)
[registry] fusion source not found: covis_v1 (exp103_covis_v1)
[registry] fusion source not found: covis_2h (exp104_covis_2hop)
[registry] fusion source not found: item2vec (exp105_item2vec)
[registry] fusion source not found: lgcn (exp106_lightgcn_import)
[registry] fusion source not found: ppr (exp107_ppr)
[registry] exp108 registered
[registry] total registered: 6


### Запуск всех зарегистрированных экспериментов

In [17]:
# Cell 5A — logging/diagnostics helpers
import os, sys, time, json, math, traceback
from collections import Counter
from pathlib import Path
from typing import Dict, List, Mapping, Optional
import numpy as np
import pandas as pd

try:
    import torch
    _HAS_TORCH = True
except Exception:
    _HAS_TORCH = False

def fmt_bytes(n):
    for unit in ["B","KB","MB","GB","TB"]:
        if n < 1024: return f"{n:.1f}{unit}"
        n /= 1024
    return f"{n:.1f}PB"

def mem_report():
    import psutil, os
    p = psutil.Process(os.getpid())
    rss = p.memory_info().rss
    out = f"RAM: {fmt_bytes(rss)}"
    if _HAS_TORCH and torch.cuda.is_available():
        try:
            alloc = torch.cuda.memory_allocated()
            res   = torch.cuda.memory_reserved()
            out += f" | GPU alloc: {fmt_bytes(alloc)}, reserved: {fmt_bytes(res)}"
        except Exception:
            pass
    return out

class Timer:
    def __init__(self, label=""):
        self.label = label
        self.t0 = time.time()
    def tick(self, msg):
        dt = time.time() - self.t0
        print(f"  [{self.label}] +{dt:.2f}s {msg}")
        self.t0 = time.time()

def describe_candidates(cmap: Dict[int, List[int]], top_items_k: int = 10) -> dict:
    n_users = len(cmap)
    covered = sum(1 for v in cmap.values() if v)
    lens = [len(v) for v in cmap.values()]
    avg_len = float(np.mean(lens)) if lens else 0.0
    p50_len = float(np.percentile(lens, 50)) if lens else 0.0
    p90_len = float(np.percentile(lens, 90)) if lens else 0.0
    cnt = Counter()
    for v in cmap.values():
        cnt.update(v[:min(100, len(v))])  # верхние позиции чаще важны
    top = cnt.most_common(top_items_k)
    return {
        "users_total": n_users,
        "users_with_candidates": covered,
        "coverage_pct": 100.0 * covered / max(1, n_users),
        "avg_list_len": avg_len,
        "p50_list_len": p50_len,
        "p90_list_len": p90_len,
        "top_items_sample": top,
    }

def print_candidates_summary(name: str, stats: dict):
    print(f"  [{name}] users={stats['users_total']} | covered={stats['users_with_candidates']} "
          f"({stats['coverage_pct']:.1f}%) | avg_len={stats['avg_list_len']:.1f} "
          f"(p50={stats['p50_list_len']:.0f}, p90={stats['p90_list_len']:.0f})")
    if stats["top_items_sample"]:
        top_str = ", ".join(f"{it}:{cnt}" for it, cnt in stats["top_items_sample"])
        print(f"  [{name}] top_items: {top_str}")

def safe_call(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        traceback.print_exc()
        raise

def pretty_print_metrics(df: Optional[pd.DataFrame], name: str):
    if df is None or len(df)==0:
        print(f"  [{name}] no metrics.")
        return
    cols = list(df.columns)
    print(f"  [{name}] metrics table ({len(df)} rows):")
    display(df[cols])

In [ ]:
# Cell 5B — verbose runner v2
from IPython.display import display

VERBOSE = True   # общий флаг болтливости
TIME_EACH_STEP = True

def run_all_verbose(REGISTRY, ctx, verbose: bool = True):
    results = {}
    print("="*80)
    print("[env] " + mem_report())
    print(f"[ctx] train_rows={len(ctx['train_df'])}, val_rows={len(ctx['val_df'])}, "
          f"date_min={ctx['split'].date_min}, train_end={ctx['split'].train_end}, "
          f"val=({ctx['split'].val_start}..{ctx['split'].val_end})")

    for key, Cls, kwargs in REGISTRY:
        print("="*80)
        print(f"▶ Running: {key}")
        t_all = Timer(key)
        try:
            exp = Cls(**(kwargs or {}))
            if verbose:
                print(f"[{key}] init done. {mem_report()}")

            # ---- FIT ----
            if TIME_EACH_STEP: t_step = Timer(f"{key}/fit")
            state = safe_call(exp.fit, ctx)
            if TIME_EACH_STEP: t_step.tick("fit() done")
            if state is not None:
                # Разные экспы имеют разные state — распечатаем характерные поля
                if hasattr(state, "embeddings_df"):
                    df = state.embeddings_df
                    print(f"  [{key}] embeddings: shape={getattr(df,'shape',None)}")
                if hasattr(state, "neighbors_df"):
                    df = state.neighbors_df
                    print(f"  [{key}] neighbors: shape={getattr(df,'shape',None)}")
                if hasattr(state, "G"):  # PPR граф
                    G = state.G
                    print(f"  [{key}] graph: U={G.U:,}, I={G.I:,}, R_nnz={G.R.nnz:,}, C_nnz={G.C.nnz:,}")
            print(f"  [{key}] after fit: {mem_report()}")

            # ---- CANDIDATES ----
            if TIME_EACH_STEP: t_step = Timer(f"{key}/candidates")
            cand_map = safe_call(exp.candidates, ctx)
            cand_stats = describe_candidates(cand_map)
            print_candidates_summary(key, cand_stats)
            if TIME_EACH_STEP: t_step.tick("candidates() done")
            print(f"  [{key}] after candidates: {mem_report()}")

            # ---- EVALUATE ----
            if TIME_EACH_STEP: t_step = Timer(f"{key}/evaluate")
            metrics = safe_call(exp.evaluate, ctx)
            pretty_print_metrics(metrics, f"{key}")
            if TIME_EACH_STEP: t_step.tick("evaluate() done")
            print(f"  [{key}] after evaluate: {mem_report()}")

            # ---- SAVE ----
            if TIME_EACH_STEP: t_step = Timer(f"{key}/save")
            paths = safe_call(exp.save, ctx)
            if isinstance(paths, (list, tuple)):
                art = [str(p) for p in paths]
            else:
                art = [str(paths)]
            print(f"  [{key}] artifacts: {art}")
            if TIME_EACH_STEP: t_step.tick("save() done")
            print(f"  [{key}] after save: {mem_report()}")

            results[key] = (exp, state, metrics, paths)
            t_all.tick("TOTAL done")
        except Exception as e:
            print(f"✗ [{key}] FAILED: {e}")
            traceback.print_exc()
            results[key] = None
            t_all.tick("FAILED")
    return results

# прогон 101–107
results = run_all_verbose([r for r in REGISTRY if r[0] != "exp108_recall_fusion"], ctx, verbose=True)

# пересобрать registry, чтобы exp108 увидел свежие candidates
# (можно просто выполнить ещё раз Cell 4)

# прогнать только fusion
only108 = [r for r in REGISTRY if r[0] == "exp108_recall_fusion"]
results_108 = run_all_verbose(only108, ctx, verbose=True)

[env] RAM: 2.3GB
[ctx] train_rows=7326431, val_rows=1451544, date_min=0, train_end=39, val=(40..46)
▶ Running: exp102_daily_trending
[exp102_daily_trending] init done. RAM: 2.3GB
✗ [exp102_daily_trending] FAILED: build_val_day_toplists() got an unexpected keyword argument 'df_all'
  [exp102_daily_trending] +0.11s FAILED
▶ Running: exp103_covis_v1
[exp103_covis_v1] init done. RAM: 2.3GB


Traceback (most recent call last):
  File "/tmp/ipykernel_165/335074487.py", line 75, in safe_call
    return fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/vseros_b/src/vseros_b/exp102_daily_trending.py", line 105, in fit
    day_top = build_val_day_toplists(
              ^^^^^^^^^^^^^^^^^^^^^^^
TypeError: build_val_day_toplists() got an unexpected keyword argument 'df_all'
Traceback (most recent call last):
  File "/tmp/ipykernel_165/2112958304.py", line 26, in run_all_verbose
    state = safe_call(exp.fit, ctx)
            ^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_165/335074487.py", line 75, in safe_call
    return fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/vseros_b/src/vseros_b/exp102_daily_trending.py", line 105, in fit
    day_top = build_val_day_toplists(
              ^^^^^^^^^^^^^^^^^^^^^^^
TypeError: build_val_day_toplists() got an unexpected keyword argument 'df_all'


  [exp103_covis_v1/fit] +270.20s fit() done
  [exp103_covis_v1] neighbors: shape=(2069257, 3)
  [exp103_covis_v1] after fit: RAM: 2.6GB


### Сабмиты для доступных экспериментов

In [ ]:
# Cell 6 — submissions (чуть шумнее)
from vseros_b.config import PATHS, COL_USER, COL_ITEM
import pandas as pd

try:
    sample = pd.read_csv(PATHS.sample_path)
    print("Sample loaded:", sample.shape)
    display(sample.head())
except Exception as e:
    print("Sample not available:", e)
    sample = None

SUB_K = 20

def try_submit(exp_key: str):
    if exp_key not in results or results[exp_key] is None or sample is None:
        print(f"[submit] {exp_key}: skipped (no results or no sample)")
        return
    exp = results[exp_key][0]
    if not hasattr(exp, "predict_submission"):
        print(f"[submit] {exp_key}: no predict_submission()")
        return
    print(f"[submit] {exp_key}: start (k={SUB_K})… {mem_report()}")
    t = Timer(f"{exp_key}/submit")
    sub_df = exp.predict_submission(ctx, sample_df=sample, k_top=SUB_K)
    t.tick("predict_submission done")
    print(f"[submit] {exp_key}: shape={sub_df.shape}")
    display(sub_df.head())

for key in ["exp101_pop_decay", "exp102_daily_trending", "exp108_recall_fusion"]:
    try_submit(key)